# Gauging stations in Catalunya
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 28-04-2026<br>

**Introduction:**<br>
This code preprocesses the discharge time series downloaded from [Agència Catalana de l'Aigua](https://analisi.transparenciacatalunya.cat/es/Medi-Ambient/Xarxes-de-control-del-medi-consulta-de-l-aigua-i-e/wc95-u57z/about_data). **The raw data includes reservoir attributes (coordinates, ID, name) and daily time series of reservoir storage, level and fraction filled. The results of the code are a CSV file with the reservoir attributes, and several CSV files (one for each reservoir) with the daily timeseries.**

In [1]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from ocab.aca import get_stations_aca

## Configuration

In [2]:
kind = 'stations'

# path where the data is stored
path_aca = Path('/home/casadoj/Data/ACA')

# paths where results will be saved
path_results = path_aca / 'processed'
path_gis = path_results / 'GIS'
path_ts = path_results / 'timeseries' / kind
for path in [path_gis, path_ts]:
    path.mkdir(parents=True, exist_ok=True)

## Extract Raw Data

Here I load and prepocess the raw Excel files downloaded from the ACA website. The result are two objects: `timeseries` is a dictionary that contains the time series for each reservoir, `reservoirs_aca` is a DataFrame with the reservoir attributes.

In [3]:
# time series will be saved in a dictionary and reservoir attributes in a pandas.DataFrame
timeseries = {}
attributes = pd.DataFrame()

# read raw Excel files iteratively
path_in = path_aca / 'raw' / kind
files = sorted(list(path_in.glob(f'{kind}_*.xlsx')))
for file in tqdm(files, desc='files'):

    # get attributes and time series
    attrs, ts = get_stations_aca(file)

    # update attributes
    attributes = pd.concat([attributes, attrs], axis=0).drop_duplicates()

    # update time series
    for ID in ts:
        if ID in timeseries:
            timeseries[ID] = pd.concat(
                (timeseries[ID], ts[ID]),
                axis=0
            ).asfreq('D').sort_index(axis=0)
        else:
            timeseries[ID] = ts[ID]

attributes.index.name = 'id_saih'

files:   0%|          | 0/8 [00:00<?, ?it/s]

## Corrections

### Attributes

Some stations are duplicated as they 

In [4]:
import matplotlib.pyplot as plt
from datetime import datetime

In [5]:
# select variable in case of multiple
for ID, ts in timeseries.items():
    if ts.shape[1] > 1:
        if 'total discharge' in ts.columns:
            timeseries[ID] = ts[['total discharge']].rename(columns={'total discharge': 'discharge'})
        elif 'discharge' in ts.columns:
            timeseries[ID] = ts[['discharge']]

In [6]:
len(timeseries), len(attributes)

(106, 106)

In [7]:
# fig, ax = plt.subplots(figsize=(16, 4))
# timeseries['EA078'].plot(ax=ax)
# ax.set_xlim('2025-01-01', '2026-01-01');
# ax.set_ylim(None, 150);

Some gauging stations are duplicated due to signals including only river streamflow, streamflow and channel flow, etc.

In [ ]:
# combine duplicates (only streamflow and streamflow+channel)
duplicates = {
    'EA066': 'EA066_C4066',
    'EA078': 'EA078_C4078',
}
for ID_keep, ID_remove in duplicates.items():
    timeseries[ID_keep] = pd.DataFrame({
    'discharge': timeseries[ID_remove]['total_discharge'].combine_first(timeseries[ID_keep]['discharge'])
})
_ = timeseries.pop(ID_remove)

# remove IDS
remove = ['EA085_C4085', 'EA120']
for ID in remove:
    _ = timeseries.pop(ID)
    
# apply changes to the attributes
attributes = attributes.loc[timeseries.keys()]

In [ ]:
# make up the ID so it's an integer and uses 9000 values as if it was in the Ebro
attributes['id'] = [int('90' + x.strip('E')) for x in attributes.index]


In [ ]:
# add attributes
attributes['basin'] = 'CATALUNYA'
for ID in attributes.index:
    ts = timeseries[ID]
    start, end = ts.index.min(), ts.index.max()
    attributes.loc[ID, ['start', 'end']] = start.year, end.year
    attributes.loc[ID, 'active'] = 1 if end.year == 2026 else 0
attributes[['start', 'end', 'active']] = attributes[['start', 'end', 'active']].astype('Int64')

# sort columns
cols = sorted([col for col in attributes.columns if col != 'geometry']) + ['geometry']
attributes = attributes[cols]

## Export

### Attributes

In [ ]:
# export
attributes.to_file(path_gis / 'dams_aca.geojson', driver='GeoJSON')

### Time series

In [ ]:
for ID, ts in timeseries.items():
    ts.to_parquet(path_ts / f'{ID}.parquet')

***